## Encoding
### Opdracht

Zoals aan het begin van de cursus besproken gaat het project over handgeschreven nummers herekennen op een MysteryDevice met de volgende eigenschappen:

- Input scherm waarmee een nieuwe plaatje als ndarray aangemaakt kan worden door de gebruiker.
- Zeer beperkt RAM (256 KB)
- Beperkte opslag (1 MB voor 
programma + model)
- Geen GPU
- Embedded python


MysteryDevice opslaglimiet: 1 MB
Volledige MNIST dataset: 52 MB

Dus het opslaan van MNST voor training doeleinden zou niet werken. Maar in ons geval maakt het ook niet uit, want we kunnen onze AI methode op een PC trainen.

Echter, hebben we nog steeds een encoding van afbeeldingen nodig omdat de gebruiker wel een afbeelding invoert.

#### Hoe ziet de input eruit tijdens inference?

De gebruiker tekent op het touchscreen → dat moet naar 28×28 of een andere encoding worden gebracht.

#### Hoeveel RAM gebruikt die inputrepresentatie?

784 bytes (1 byte per pixel) -> heel laag
maar 32‑bits floats → 784 × 4 = 3 KB -> al een stuk hoger

Ontwerp een encoding van een MNIST plaatje die zo min mogelijk geheugen nodig heeft op het MysteryDevice om je Decision Tree uit de vorige opdracht te draaien op 1 sample. 

Mogelijk heb je al wat gedaan in de vorige stap, maar kijk of je het nog meer kan verbeteren.
Het moet:

- minder geheugen gebruiken
- minder rekenkracht nodig hebben
- toch informatie bewaren om cijfers te herkennen

Maak eventueel gebruik aan de volgende onderweren, maar ga vooral zelf experimenteren:
- binning
(“donker / medium / licht” → 3 waarden)

- binary thresholding
(zwart/wit → 1 bit per pixel)

- downscaling
(28×28 → 14×14 → 196 features)

- flattening
(784 → 784 vector)

- quantization
(8-bit → 4-bit → 2-bit per pixel)

- feature extraction
(bijv. “hoeveel inkt zit links/rechts/boven/onder?”)
- sparse encoding
(alleen niet‑nul pixels bewaren)

- Ga je normaliseren? Hoe?
- Ga je flattenen of 2D laten staan?
- Ga je binning toepassen op pixelwaarden?
- Ga je ordinale encoding toepassen op pixelintensiteiten?
- Ga je pixelwaarden reduceren (bijv. “donker, medium, licht”)?
- Ga je één pixel gebruiken, of alle 784?

**Ga ook een eigen “encoding” bedenken!**

In [ ]:
import numpy as np
from PIL import Image

def encode(img):
    pil = Image.fromarray(img.astype(np.uint8))
    small = np.array(pil.resize((14, 14), Image.LANCZOS), dtype=np.uint8)
    
    binary = (small > 128).astype(np.uint8)
    
    return np.packbits(binary.flatten())

encoded_sample = encode(sample)
encoded_sample.nbytes

Beantwoord ook de volgende vragen:

1. Hoeveel RAM kost één afbeelding?
2. Hoeveel RAM kost 100 afbeeldingen?
3. Hoe groot zou het model maximaal mogen zijn?
4. Kun je de encoding verder comprimeren?
5. Is er informatie verloren gegaan?
6. Kun je nog steeds het cijfer herkennen?

In [ ]:
print(f"1 afbeelding: {encoded_sample.nbytes} bytes")

In [ ]:
print(f"100 afbeeldingen: {encoded_sample.nbytes * 100} bytes ({encoded_sample.nbytes * 100 / 1024:.2f} KB)")

In [ ]:
ram_totaal = 256 * 1024  # 256 KB in bytes
overhead = 50 * 1024     # geschatte Python runtime overhead
sample_ram = encoded_sample.nbytes

model_max = ram_totaal - overhead - sample_ram
print(f"RAM totaal:      {ram_totaal} bytes ({ram_totaal/1024:.0f} KB)")
print(f"Overhead:        {overhead} bytes ({overhead/1024:.0f} KB)")
print(f"Sample:          {sample_ram} bytes")
print(f"Model max:       {model_max} bytes ({model_max/1024:.1f} KB)")

In [ ]:
import zlib

encoded_bytes = encoded_sample.tobytes()
compressed = zlib.compress(encoded_bytes)
print(f"Na packbits:   {len(encoded_bytes)} bytes")
print(f"Na zlib:       {len(compressed)} bytes")
print(f"Winst:         {len(encoded_bytes) - len(compressed)} bytes")

In [ ]:
import matplotlib.pyplot as plt

origineel = X_train[0]
pil = Image.fromarray(origineel.astype(np.uint8))
small = np.array(pil.resize((14, 14), Image.LANCZOS), dtype=np.uint8)
binary = (small > 128).astype(np.uint8)

fig, axes = plt.subplots(1, 3, figsize=(8, 3))
axes[0].imshow(origineel, cmap='gray')
axes[0].set_title("Origineel (28×28)")
axes[1].imshow(small, cmap='gray')
axes[1].set_title("Downscaled (14×14)")
axes[2].imshow(binary, cmap='gray')
axes[2].set_title("Binair (14×14)")
for ax in axes:
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# Vergelijk voorspelling van het origineel vs de encoded versie
from sklearn.tree import DecisionTreeClassifier

# Train op originele pixels
clf_orig = DecisionTreeClassifier(max_depth=8)
clf_orig.fit(X_train[:5000].reshape(5000, -1), y_train[:5000])

# Train op encoded pixels
X_encoded = np.array([np.unpackbits(np.packbits((np.array(Image.fromarray(x.astype(np.uint8)).resize((14,14), Image.LANCZOS)) > 128).astype(np.uint8).flatten()))[:196] for x in X_train[:5000]])
clf_enc = DecisionTreeClassifier(max_depth=8)
clf_enc.fit(X_encoded, y_train[:5000])

X_test_encoded = np.array([np.unpackbits(np.packbits((np.array(Image.fromarray(x.astype(np.uint8)).resize((14,14), Image.LANCZOS)) > 128).astype(np.uint8).flatten()))[:196] for x in X_test[:1000]])

acc_orig = clf_orig.score(X_test[:1000].reshape(1000, -1), y_test[:1000])
acc_enc  = clf_enc.score(X_test_encoded, y_test[:1000])

print(f"Accuracy origineel: {acc_orig*100:.1f}%")
print(f"Accuracy encoded:   {acc_enc*100:.1f}%")